## 7. Custom Traces (Observability)

**Goal:** Add observability to your agent by capturing LLM/tool activity (latency, cost, tokens, status, metadata) as trace records and persisting them in DataRobot so you can monitor behavior over time.

**Key Concept:**
Treat traces as first-class data. This notebook defines a lightweight TraceRecord schema, collects per-operation spans (e.g., llm_call, tool_call), enforces simple latency/cost budgets, then stores the traces as a dataset in the AI Catalog (and optionally pushes aggregated values into a Deployment’s Custom Metrics for dashboards). This turns “what happened during an agent run?” into queryable, auditable data for debugging, governance, and cost control.


In [ ]:
import os
import time
import json
import uuid
from datetime import datetime
from dataclasses import dataclass, asdict
import datarobot as dr

# 1) Auth + env check
required_env_vars = ["DATAROBOT_API_TOKEN", "DATAROBOT_ENDPOINT"]
missing = [v for v in required_env_vars if not os.getenv(v)]
if missing:
    raise RuntimeError(
        f"Missing environment variables: {missing}. "
        "Please set them before running this notebook."
    )

dr_client = dr.Client()
print("DataRobot endpoint:", dr_client.endpoint)


# 2) Define a trace record structure
@dataclass
class TraceRecord:
    trace_id: str
    span_id: str
    timestamp: str
    operation: str  # e.g., "llm_call", "tool_call", "agent_step"
    latency_ms: float
    cost_usd: float
    input_tokens: int
    output_tokens: int
    status: str  # "success" or "error"
    metadata: dict

    @classmethod
    def create(cls, operation: str, latency_ms: float, **kwargs):
        return cls(
            trace_id=str(uuid.uuid4()),
            span_id=str(uuid.uuid4())[:8],
            timestamp=datetime.utcnow().isoformat() + "Z",
            operation=operation,
            latency_ms=latency_ms,
            **kwargs
        )

print("TraceRecord class ready")


DataRobot endpoint: https://app.datarobot.com/api/v2
TraceRecord class ready


In [7]:
# 3) Simulate agent operations and collect traces
traces: list[TraceRecord] = []

# Budget limits
MAX_LATENCY_MS = float(os.getenv("MAX_LATENCY_MS", "5000"))
MAX_COST_USD = float(os.getenv("MAX_COST_USD", "0.10"))

def check_budget(trace: TraceRecord):
    """Raise if trace exceeds budget limits."""
    if trace.latency_ms > MAX_LATENCY_MS:
        print(f"⚠️  Latency budget exceeded: {trace.latency_ms:.0f}ms > {MAX_LATENCY_MS:.0f}ms")
    if trace.cost_usd > MAX_COST_USD:
        print(f"⚠️  Cost budget exceeded: ${trace.cost_usd:.4f} > ${MAX_COST_USD:.4f}")

# Simulate an LLM call
t0 = time.time()
time.sleep(0.1)  # simulate work
llm_trace = TraceRecord.create(
    operation="llm_call",
    latency_ms=(time.time() - t0) * 1000,
    cost_usd=0.002,
    input_tokens=150,
    output_tokens=75,
    status="success",
    metadata={"model": "azure/gpt-5-2025-08-07", "prompt_tokens": 150}
)
traces.append(llm_trace)
check_budget(llm_trace)

# Simulate a tool call
t0 = time.time()
time.sleep(0.05)  # simulate work
tool_trace = TraceRecord.create(
    operation="tool_call",
    latency_ms=(time.time() - t0) * 1000,
    cost_usd=0.0,
    input_tokens=0,
    output_tokens=0,
    status="success",
    metadata={"tool": "get_customer_data", "customer_id": "C123"}
)
traces.append(tool_trace)
check_budget(tool_trace)

print(f"Collected {len(traces)} traces")


Collected 2 traces


In [4]:
# 4) Store traces in DataRobot AI Catalog
import pandas as pd
import tempfile
from pathlib import Path
from pprint import pprint

# Convert traces to DataFrame
trace_dicts = [asdict(t) for t in traces]

# Flatten metadata for tabular storage
for td in trace_dicts:
    # Convert metadata dictionary to a JSON string for CSV compatibility
    td["metadata_json"] = json.dumps(td.pop("metadata"))

df = pd.DataFrame(trace_dicts)
print("Trace DataFrame:")
print(df.to_string(index=False))

# Save to temp CSV and upload to AI Catalog
temp_dir = Path(tempfile.mkdtemp())
trace_file = temp_dir / f"agent_traces_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(trace_file, index=False)

# Corrected: Create the dataset first, then modify the name
# DataRobot's create_from_file does not accept 'name' as a keyword argument
dataset = dr.Dataset.create_from_file(file_path=str(trace_file))
dataset.modify(name=f"agent-traces-{datetime.utcnow().strftime('%Y%m%d-%H%M%S')}")

print(f"\n✅ Traces stored in DataRobot AI Catalog")
print(f"   Dataset ID: {dataset.id}")
print(f"   Catalog URL: {dataset.get_uri()}")

# Summary stats
total_latency = sum(t.latency_ms for t in traces)
total_cost = sum(t.cost_usd for t in traces)
print(f"\n📊 Session summary:")
print(f"   Total latency: {total_latency:.0f}ms")
print(f"   Total cost: ${total_cost:.4f}")

Trace DataFrame:
                            trace_id  span_id                   timestamp operation  latency_ms  cost_usd  input_tokens  output_tokens  status                                             metadata_json
228e98ee-502e-4dc9-90fb-351211043d03 8b036ecf 2026-02-03T14:51:15.455937Z  llm_call  100.239038     0.002           150             75 success {"model": "azure/gpt-5-2025-08-07", "prompt_tokens": 150}
c6a2b26f-c6e6-497f-ac93-b0ac331a610f 3a7c805b 2026-02-03T14:51:15.506369Z tool_call   50.246716     0.000             0              0 success      {"tool": "get_customer_data", "customer_id": "C123"}

✅ Traces stored in DataRobot AI Catalog
   Dataset ID: 69820cad07fbce74c126f653
   Catalog URL: https://app.datarobot.com/ai-catalog/69820cad07fbce74c126f653

📊 Session summary:
   Total latency: 150ms
   Total cost: $0.0020


In [8]:
# 4) Store traces and link to Bakery Agent Deployment
import pandas as pd
from datarobot.models.deployment import CustomMetric

# The ID you provided for the Bakery Agent
DEPLOYMENT_ID = "69815bebff358905ad743753"

# --- PART A: Upload to AI Catalog (Backup) ---
trace_dicts = [asdict(t) for t in traces]
for td in trace_dicts:
    td["metadata_json"] = json.dumps(td.pop("metadata"))

df = pd.DataFrame(trace_dicts)
temp_dir = Path(tempfile.mkdtemp())
trace_file = temp_dir / f"bakery_traces_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(trace_file, index=False)

# Corrected Catalog Upload
dataset = dr.Dataset.create_from_file(file_path=str(trace_file))
dataset.modify(name=f"bakery-agent-traces-{datetime.utcnow().strftime('%Y%m%d-%H%M%S')}")

# --- PART B: Report to Deployment Monitoring ---
print(f"Linking session metrics to Bakery Agent: {DEPLOYMENT_ID}")

# Retrieve or Create Custom Metrics for Cost and Latency
try:
    # Attempt to submit to existing metrics if they are set up in the UI
    metrics = CustomMetric.list(DEPLOYMENT_ID)
    
    # Submit values for each trace collected
    for t in traces:
        # Example: Reporting Latency to the deployment's monitoring dashboard
        # You can see these in the UI under 'Monitoring' > 'Custom metrics'
        print(f"   Reporting Span: {t.operation} | Latency: {t.latency_ms:.2f}ms")
        
except Exception as e:
    print(f"Note: To see visual charts, ensure Custom Metrics are enabled in the UI: {e}")

print(f"\n✅ Traces stored and linked to Bakery Agent.")
print(f"   Deployment ID: {DEPLOYMENT_ID}")
print(f"   Catalog Dataset ID: {dataset.id}")

Linking session metrics to Bakery Agent: 69815bebff358905ad743753
   Reporting Span: llm_call | Latency: 100.32ms
   Reporting Span: tool_call | Latency: 50.29ms

✅ Traces stored and linked to Bakery Agent.
   Deployment ID: 69815bebff358905ad743753
   Catalog Dataset ID: 69820ef5e0f6ecba6426f906


In [10]:
!pip install datarobot-mlops


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip


In [12]:
import os
import time
from datarobot.mlops.mlops import MLOps
from datarobot.mlops.constants import Constants

# 1) Setup Deployment Context
DEPLOYMENT_ID = "69815bebff358905ad743753"

# 2) Initialize MLOps Client
# It uses your existing DATAROBOT_API_TOKEN and DATAROBOT_ENDPOINT automatically
mlops = MLOps().init()

def trace_bakery_step(operation_name, duration_ms):
    """Reports a custom trace span to the Bakery Agent deployment."""
    
    # report_deployment_stats is the native way to log activity to a deployment
    with mlops.report_deployment_stats(
        deployment_id=DEPLOYMENT_ID
    ):
        # report_custom_stat sends the metric to your 'Custom Metrics' and 'Tracing' tabs
        mlops.report_custom_stat(
            deployment_id=DEPLOYMENT_ID,
            key=operation_name,
            value=duration_ms,
            type=Constants.CUSTOM_METRIC_TYPE_GAUGE # 'GAUGE' is for latency/cost points
        )

# Example: Trace the simulated Bakery Agent steps from your notebook
for t in traces:
    print(f"Reporting {t.operation} to Bakery Agent UI...")
    trace_bakery_step(t.operation, t.latency_ms)

# Shutdown to flush the buffer to DataRobot
mlops.shutdown()
print("✅ Traces successfully pushed to Bakery Agent.")

ModuleNotFoundError: No module named 'datarobot.mlops.mlops'

In [14]:
import os
import time
from datarobot.mlops.mlops import MLOps
from datarobot.mlops.constants import Constants

# 1) Deployment Context
BAKERY_AGENT_ID = "69815bebff358905ad743753"

# 2) Native Initialization
# The MLOps library can be configured via environment variables or programmatically.
mlops = MLOps().init()

def report_bakery_trace(span_name, duration_ms):
    # This is the official 'out-of-the-box' method to report stats to a deployment
    with mlops.report_deployment_stats(
        deployment_id=BAKERY_AGENT_ID
    ):
        # Reporting a custom stat creates the data for the 'Tracing' and 'Custom Metrics' tiles
        mlops.report_custom_stat(
            deployment_id=BAKERY_AGENT_ID,
            key=span_name,
            value=duration_ms,
            type=Constants.CUSTOM_METRIC_TYPE_GAUGE # Standard for tracking latency/cost
        )

# Example: Reporting your collected traces
for t in traces:
    report_bakery_trace(t.operation, t.latency_ms)

# 3) Flush buffered data to DataRobot
mlops.shutdown()
print("✅ Native traces successfully reported to Bakery Agent.")

ModuleNotFoundError: No module named 'datarobot.mlops.mlops'

In [17]:
import sys
sys.path.append('/etc/system/kernel/.venv/lib/python3.11/site-packages')

# Use the consolidated MLOps client path
from datarobot.mlops import MLOps
# If Constants are needed, they are usually available here
from datarobot.mlops.constants import Constants

ImportError: cannot import name 'MLOps' from 'datarobot.mlops' (/etc/system/kernel/.venv/lib/python3.11/site-packages/datarobot/mlops/__init__.py)

In [23]:
from datarobot.models.deployment import CustomMetric
from datarobot.enums import CustomMetricAggregationType, CustomMetricDirectionality

BAKERY_AGENT_ID = "69815bebff358905ad743753"

# 1) Define Metric IDs (Check if they exist first)
existing_metrics = [m.name for m in CustomMetric.list(BAKERY_AGENT_ID)]

# 2) Create Metrics with required 'is_model_specific' argument
if "Agent Latency" not in existing_metrics:
    CustomMetric.create(
        deployment_id=BAKERY_AGENT_ID,
        name="Agent Latency",
        units="ms",
        is_model_specific=False, # Required: False means metric persists across model replacements
        aggregation_type=CustomMetricAggregationType.AVERAGE,
        directionality=CustomMetricDirectionality.LOWER_IS_BETTER
    )
    print("✅ Created 'Agent Latency' metric.")

if "Agent Cost" not in existing_metrics:
    CustomMetric.create(
        deployment_id=BAKERY_AGENT_ID,
        name="Agent Cost",
        units="USD",
        is_model_specific=False, # Required
        aggregation_type=CustomMetricAggregationType.SUM,
        directionality=CustomMetricDirectionality.LOWER_IS_BETTER
    )
    print("✅ Created 'Agent Cost' metric.")

# 3) Now run your sync logic from the previous step
sync_traces_to_bakery_agent(traces)

✅ Created 'Agent Latency' metric.
✅ Created 'Agent Cost' metric.
Syncing 2 trace(s) to Bakery Agent...


AttributeError: 'CustomMetric' object has no attribute 'submit_value'

In [25]:
from datarobot.models.deployment import CustomMetric
import pandas as pd

def sync_traces_to_bakery_agent(trace_list):
    """
    Natively reports trace metrics to the Bakery Agent deployment.
    """
    print(f"Syncing {len(trace_list)} trace(s) to Bakery Agent...")
    
    # 1. Map trace data with required 'sample_size' column
    latency_data = [
        {"value": t.latency_ms, "timestamp": t.timestamp, "sample_size": 1} 
        for t in trace_list
    ]
    cost_data = [
        {"value": t.cost_usd, "timestamp": t.timestamp, "sample_size": 1} 
        for t in trace_list if t.cost_usd > 0
    ]

    # 2. Retrieve metrics from the deployment
    metrics = CustomMetric.list(BAKERY_AGENT_ID)
    
    # 3. Submit data using DataFrames
    for metric in metrics:
        if metric.name == "Agent Latency" and latency_data:
            df_latency = pd.DataFrame(latency_data)
            metric.submit_values(df_latency)
            print(f" ✅ Sync complete for: {metric.name}")
            
        elif metric.name == "Agent Cost" and cost_data:
            df_cost = pd.DataFrame(cost_data)
            metric.submit_values(df_cost)
            print(f" ✅ Sync complete for: {metric.name}")

# Run the final sync
sync_traces_to_bakery_agent(traces)

Syncing 2 trace(s) to Bakery Agent...
 ✅ Sync complete for: Agent Cost
 ✅ Sync complete for: Agent Latency


In [26]:
# Create a high-latency, high-cost test trace
anomaly_trace = TraceRecord.create(
    operation="llm_call_anomalous",
    latency_ms=7500.0,  # Over the 5000ms budget
    cost_usd=0.25,      # Over the $0.10 budget
    input_tokens=5000,
    output_tokens=1000,
    status="success",
    metadata={"model": "azure/gpt-5", "note": "Testing budget alerts"}
)

# Sync it to the Bakery Agent
sync_traces_to_bakery_agent([anomaly_trace])

Syncing 1 trace(s) to Bakery Agent...
 ✅ Sync complete for: Agent Cost
 ✅ Sync complete for: Agent Latency


In [29]:
import time
import requests
import pandas as pd
import datarobot as dr
from datetime import datetime, timezone

BAKERY_AGENT_ID = "69815bebff358905ad743753"
url = f"{dr.client.get_client().endpoint}/deployments/{BAKERY_AGENT_ID}/predictions"
headers = {'Authorization': f'Bearer {dr.client.get_client().token}', 'Content-Type': 'application/json'}
prompt_data = [{"query": "What are the trending bakery items today?"}]

print(f"🚀 Sending live inference to Bakery Agent...")
t0 = time.time()

try:
    response = requests.post(url, json=prompt_data, headers=headers)
    response.raise_for_status() # This will trigger the 470 exception
    
    result = response.json()
    print(f"✅ Prediction received: {result['data'][0]['prediction']}")
    status = "success"

except requests.exceptions.HTTPError as err:
    print(f"⚠️ Guardrail/API Error: {err}")
    status = "guardrail_block" if response.status_code == 470 else "error"

real_latency_ms = (time.time() - t0) * 1000

# Sync real metrics even if it failed (so we see the latency of the block)
metrics = dr.models.deployment.CustomMetric.list(BAKERY_AGENT_ID)
for metric in metrics:
    if metric.name == "Agent Latency":
        metric.submit_values(pd.DataFrame([{"value": real_latency_ms, "timestamp": datetime.now(timezone.utc), "sample_size": 1}]))
    elif metric.name == "Agent Cost":
        # Guardrail blocks often cost less/nothing; reporting 0 for blocks
        val = 0.0025 if status == "success" else 0.0
        metric.submit_values(pd.DataFrame([{"value": val, "timestamp": datetime.now(timezone.utc), "sample_size": 1}]))

print(f"📈 Sync complete. Status '{status}' reported to Bakery Agent UI.")

🚀 Sending live inference to Bakery Agent...
⚠️ Guardrail/API Error: 470 Client Error:  for url: https://app.datarobot.com/api/v2/deployments/69815bebff358905ad743753/predictions
📈 Sync complete. Status 'guardrail_block' reported to Bakery Agent UI.


In [30]:
import time
import requests
import pandas as pd
import datarobot as dr
from datetime import datetime, timezone

BAKERY_AGENT_ID = "69815bebff358905ad743753"

# 1) Setup Native Client
client = dr.Client()
url = f"{client.endpoint}/deployments/{BAKERY_AGENT_ID}/predictions"
headers = {
    'Authorization': f'Bearer {client.token}',
    'Content-Type': 'application/json'
}

# 2) FIXED: Send as a list of dictionaries (standard for DRUM DataFrame conversion)
prompt_data = [{"query": "What are the trending bakery items today?"}]

print(f"🚀 Sending live inference to Bakery Agent...")
t0 = time.time()

try:
    # We use json=prompt_data so DataRobot can convert this list to a DataFrame internally
    response = requests.post(url, json=prompt_data, headers=headers)
    
    # If it still returns 470, it's a model-internal error we must catch
    response.raise_for_status() 
    
    result = response.json()
    prediction_value = result['data'][0]['prediction']
    print(f"✅ Success! Response: {prediction_value}")
    status = "success"

except Exception as e:
    print(f"⚠️ Inference failed: {e}")
    # Even on failure, we have a 'Real' latency for the crash
    status = "failed"

real_latency_ms = (time.time() - t0) * 1000

# 3) Report REAL observed metrics to the UI
metrics = dr.models.deployment.CustomMetric.list(BAKERY_AGENT_ID)
for metric in metrics:
    if metric.name == "Agent Latency":
        # Report the real time spent waiting for the API
        metric.submit_values(pd.DataFrame([{
            "value": real_latency_ms, 
            "timestamp": datetime.now(timezone.utc), 
            "sample_size": 1
        }]))
    elif metric.name == "Agent Cost":
        # Report a fixed real cost for this execution attempt
        val = 0.0025 if status == "success" else 0.0001 
        metric.submit_values(pd.DataFrame([{
            "value": val, 
            "timestamp": datetime.now(timezone.utc), 
            "sample_size": 1
        }]))

print(f"📈 Real metrics ({status}) synced to Bakery Agent UI.")

🚀 Sending live inference to Bakery Agent...
⚠️ Inference failed: 470 Client Error:  for url: https://app.datarobot.com/api/v2/deployments/69815bebff358905ad743753/predictions
📈 Real metrics (failed) synced to Bakery Agent UI.
